<a href="https://colab.research.google.com/github/hoanganh1105/ai-ambulance-coordinator/blob/main/notebooks/Main_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Mục lục

>[Cài đặt](#scrollTo=JFhQ6i_J1w2X)

>[Thử nghiệm từng module](#scrollTo=FWN6Zs5HfBDl)

>>>[Dự báo tình trạng giao thông](#scrollTo=Ykbuy9J8g070)

>>>[Chẩn đoán bệnh](#scrollTo=7QGIlJ0Ug37w)

>>>[Ưu tiên bệnh nhân](#scrollTo=GN9rxcBDiGfm)

>>>[Tìm đường tới bệnh nhân được ưu tiên](#scrollTo=o24N4W48ibuD)

>[Hệ thống tích hợp](#scrollTo=1jifPjlXHcPl)

>>>[Khởi tạo mô hình](#scrollTo=OfkbRTyz199c)

>>>[Cập nhật môi trường](#scrollTo=9h6zxO28KKek)

>>>[Xử lý chính](#scrollTo=L-2gWIeCZYaw)

>>>[Giao diện tương tác](#scrollTo=Wy6ZwBZ_a5c4)



# Cài đặt

Clone thư mục từ github, cài đặt thư viện và import các module.

In [ ]:
!git clone https://github.com/hoanganh1105/ai-ambulance-coordinator.git
%cd ai-ambulance-coordinator
!pip install -r requirements.txt

from tabulate import tabulate
from modules.core.disease_classifier import *
from modules.core.map_router import *
from modules.core.patient_prioritizer import *
from modules.core.traffic_estimator import *

Tải dataset để train các mô hình:
- [Disease-Symptom Dataset](https://www.kaggle.com/datasets/dhivyeshrk/diseases-and-symptoms-dataset)
- [Delhi Traffic Patterns Dataset](https://www.kaggle.com/datasets/guriya79/understanding-delhi-traffic-patterns)

In [ ]:
import kagglehub
disease_classifier_train_dataset = kagglehub.dataset_download("dhivyeshrk/diseases-and-symptoms-dataset", path="Final_Augmented_dataset_Diseases_and_Symptoms.csv")
print("Path to disease dataset:", disease_classifier_train_dataset)
traffic_estimator_train_dataset = kagglehub.dataset_download("guriya79/understanding-delhi-traffic-patterns", path="delhi_traffic_features.csv")
print("Path to traffic dataset:", traffic_estimator_train_dataset)

# Thử nghiệm từng module

### Dự báo tình trạng giao thông

In [ ]:
def test_traffic_estimator():
  # Khởi tạo mô hình
  traffic_est = TrafficEstimator()
  traffic_est.train(traffic_estimator_train_dataset)
  router = MapRouter()
  # Cấu hình môi trường
  time_of_day = "Night"           # "Morning Peak" / "Afternoon" / "Evening Peak" / "Night"
  day_of_week = "Weekday"         # "Weekday" / "Weekend"
  weather_condition = "Rain"      # "Clear" / "Rain" / "Fog" / "Heatwave"

  # Hàm cập nhật trọng số.
  def weight_func(data, alpha = 0.5):
    road_type = data.get("highway")
    jam_level = traffic_est.estimate_traffic_density_level(time_of_day,
                                                           day_of_week,
                                                           weather_condition,
                                                           road_type)
    length = data.get("length")
    data["density_level"] = jam_level
    return length * (1 + alpha * jam_level)

  router.add_edges_attribute("weight", weight_func)
  router.show_map(show_density=True)

test_traffic_estimator()

### Chẩn đoán bệnh

In [ ]:
def test_disease_classifier():
  import time
  import tracemalloc
  import random
  import warnings
  import pandas as pd
  import matplotlib.pyplot as plt
  from IPython.display import display, HTML
  # ==============================================================================
  # PHẦN 1: CẤU HÌNH THỰC NGHIỆM (CONFIGURATION)
  # ==============================================================================
  DATASET_PATH = disease_classifier_train_dataset
  NUM_INFERENCE_QUERIES = 1000  # Số lượng bệnh nhân giả lập để stress test

  print("\n🚀 KHỞI TẠO PIPELINE THỰC NGHIỆM VÀ ĐÁNH GIÁ HIỆU NĂNG...")
  print("-" * 65)

  # Khởi tạo mô hình
  clf_sklearn = DiseaseClassifier(model_name="sklearnNaiveBayes")
  clf_simple = DiseaseClassifier(model_name="simpleClassifierModel")

  benchmark_results = {
      "Metrics": ["Training Time (sec)", "Peak Memory (MB)", "Inference Time - 1000 Queries (sec)", "Accuracy (%)"]
  }

  # ==============================================================================
  # PHẦN 2: ĐO LƯỜNG HUẤN LUYỆN & BỘ NHỚ (TRAINING BENCHMARK)
  # ==============================================================================
  print("[1/4] Đang huấn luyện và đo lường Scikit-Learn...")
  tracemalloc.start()
  start_t = time.time()
  clf_sklearn.train(DATASET_PATH)
  time_sk = time.time() - start_t
  _, peak_mem_sk = tracemalloc.get_traced_memory()
  tracemalloc.stop()

  print("[2/4] Đang huấn luyện và đo lường NumPy (From Scratch)...")
  tracemalloc.start()
  start_t = time.time()
  clf_simple.train(DATASET_PATH)
  time_sim = time.time() - start_t
  _, peak_mem_sim = tracemalloc.get_traced_memory()
  tracemalloc.stop()

  # Chuyển đổi Memory sang MB
  mem_sk_mb = peak_mem_sk / (1024 * 1024)
  mem_sim_mb = peak_mem_sim / (1024 * 1024)

  # ==============================================================================
  # PHẦN 3: ĐÁNH GIÁ ĐỘ CHÍNH XÁC & TEST DỰ ĐOÁN THỰC TẾ
  # ==============================================================================
  print("[3/4] Đang đánh giá chỉ số và chạy Test dự đoán thực tế...")
  eval_sk = clf_sklearn.evaluate()
  eval_sim = clf_simple.evaluate()

  acc_sk = eval_sk.get('Accuracy', eval_sk.get('accuracy', 0)) * 100
  acc_sim = eval_sim.get('Accuracy', eval_sim.get('accuracy', 0)) * 100

  # Lấy tự động 2 triệu chứng đầu tiên để test sanity check
  sample_symptoms = [clf_sklearn.symptoms_vocab[0], clf_sklearn.symptoms_vocab[5]]
  print(f"\n   -> Triệu chứng test: {sample_symptoms}")
  print(f"   -> Kết quả Scikit-Learn: {clf_sklearn.predict(sample_symptoms)}")
  print(f"   -> Kết quả NumPy:        {clf_simple.predict(sample_symptoms)}\n")

  # ==============================================================================
  # PHẦN 4: STRESS TEST TỐC ĐỘ SUY DIỄN (INFERENCE BENCHMARK)
  # ==============================================================================
  print(f"[4/4] Đang Stress Test tốc độ suy diễn ({NUM_INFERENCE_QUERIES} truy vấn)...")
  vocab = list(clf_sklearn.symptoms_vocab)

  # Tạo dữ liệu ảo
  synthetic_patients = [random.sample(vocab, random.randint(3, 6)) for _ in range(NUM_INFERENCE_QUERIES)]

  start_t = time.time()
  for patient in synthetic_patients:
      _ = clf_sklearn.predict(patient)
  inf_sk = time.time() - start_t

  start_t = time.time()
  for patient in synthetic_patients:
      _ = clf_simple.predict(patient)
  inf_sim = time.time() - start_t

  # ==============================================================================
  # PHẦN 5: XUẤT BÁO CÁO & VẼ BIỂU ĐỒ TRỰC QUAN
  # ==============================================================================
  print("\n✅ HOÀN TẤT THỰC NGHIỆM! ĐANG TẠO BÁO CÁO...\n")

  benchmark_results["Scikit-Learn (Scale-up)"] = [
      round(time_sk, 4), round(mem_sk_mb, 2), round(inf_sk, 4), round(acc_sk, 2)
  ]
  benchmark_results["NumPy (From Scratch)"] = [
      round(time_sim, 4), round(mem_sim_mb, 2), round(inf_sim, 4), round(acc_sim, 2)
  ]

  # Tạo DataFrame
  df_report = pd.DataFrame(benchmark_results)
  df_report.set_index("Metrics", inplace=True)

  # Hiển thị bảng
  display(HTML("<h3 style='color:#2E86C1;'>📊 Báo Cáo Thực Nghiệm Hiệu Năng (Benchmark Report)</h3>"))
  display(df_report.style.highlight_min(axis=1, color='lightgreen', subset=pd.IndexSlice[["Training Time (sec)", "Peak Memory (MB)", "Inference Time - 1000 Queries (sec)"], :]) \
                  .highlight_max(axis=1, color='lightgreen', subset=pd.IndexSlice[["Accuracy (%)"], :]) \
                  .format(precision=4))

  # Vẽ biểu đồ 1
  fig1, ax1 = plt.subplots(figsize=(7, 5))
  bars1 = ax1.bar(["Scikit-Learn", "NumPy (Scratch)"], [inf_sk, inf_sim], color=['#3498DB', '#E74C3C'])
  ax1.set_title(f"Thời gian chẩn đoán {NUM_INFERENCE_QUERIES} ca bệnh\n(Thấp hơn là Tốt hơn)")
  ax1.set_ylabel("Giây (Seconds)")
  for bar in bars1:
      yval = bar.get_height()
      ax1.text(bar.get_x() + bar.get_width()/2, yval + 0.01, f"{yval:.4f}s", ha='center', va='bottom', fontweight='bold')
  plt.tight_layout()

  # Vẽ biểu đồ 2
  fig2, ax2 = plt.subplots(figsize=(7, 5))
  bars2 = ax2.bar(["Scikit-Learn", "NumPy (Scratch)"], [mem_sk_mb, mem_sim_mb], color=['#2ECC71', '#F39C12'])
  ax2.set_title("Bộ nhớ RAM tối đa khi Huấn luyện\n(Thấp hơn là Tốt hơn)")
  ax2.set_ylabel("Megabytes (MB)")
  for bar in bars2:
      yval = bar.get_height()
      ax2.text(bar.get_x() + bar.get_width()/2, yval + 0.5, f"{yval:.2f} MB", ha='center', va='bottom', fontweight='bold')
  plt.tight_layout()

  plt.show()

test_disease_classifier()

### Ưu tiên bệnh nhân

In [ ]:
def test_patient_prioritizer():
  prioritizer = PatientPrioritizer()
  prioritizer.load_knowledge_base("features/knowledgebase.txt")
  # Tạo danh sách bệnh nhân giả lập
  patient_list = []

  # Bệnh nhân 1
  p1 = Patient(["frontal headache", "dizziness"], (0, 0))
  p1.predicted_disease = "benign paroxysmal positional vertical (bppv)"
  patient_list.append(p1)

  # Bệnh nhân 2
  p2 = Patient(["sore throat", "hoarse voice", "difficulty breathing"], (0, 0))
  p2.predicted_disease = "head and neck cancer"
  patient_list.append(p2)

  # Bệnh nhân 3
  p3 = Patient(["bleeding from eye", "bleeding from ear", "fainting"], (0, 0))
  p3.predicted_disease = "hypovolemia"
  patient_list.append(p3)

  # Bệnh nhân 4
  p4 = Patient(["fainting", "feeling ill", "vomiting blood"], (0, 0))
  p4.predicted_disease = "hypovolemia"
  patient_list.append(p4)

  print("Danh sách bệnh nhân ban đầu:\n")
  headers = ["ID", "Triệu chứng", "Chẩn đoán"]
  rows = [[p.id, ", ".join(p.symptoms), p.predicted_disease] for p in patient_list]
  print(tabulate(rows, headers=headers, tablefmt="grid"))

  most_prior_patients = prioritizer.get_most_prioritized_patients(patient_list)
  print(f"\nID bệnh nhân được ưu tiên: {[p.id for p in most_prior_patients]}")

test_patient_prioritizer()

### Tìm đường tới bệnh nhân được ưu tiên

In [ ]:
def test_route_planner():
  print("--- Đang khởi tạo MapRouter (Simple Street Map) ---")
  # Khởi tạo router cho khu vực Delhi
  router = MapRouter(place_name="Delhi, India", model_name="simpleStreetMap")

  # 1. Lấy danh sách toạ độ khả dụng
  coords = router.available_coordinates()
  if not coords:
      print("Không tìm thấy dữ liệu bản đồ!")
      return

  print(f"Số lượng node trên bản đồ: {len(coords)}")

  # 2. Giả định toạ độ xuất phát (Ambulance) và đích (Patient)
  origin_coords = coords[100] # Lấy một điểm ở giữa danh sách để test rõ hơn
  target_coords = coords[-100]

  print(f"Vị trí org: {origin_coords}")
  print(f"Vị trí target: {target_coords}")

  # 3. Tìm đường đi tối ưu
  print("\n--- Đang tìm đường đi tối ưu... ---")
  try:
      # SỬA LỖI TẠI ĐÂY: optimal_path trả về (path_coords, distance)
      path_coords, distance = router.optimal_path(origin_coords, target_coords)
      print(f"Tìm thấy đường đi với {len(path_coords)} toạ độ.")
      print(f"Tổng chiều dài quãng đường: {distance:.2f} meters")
  except Exception as e:
      print(f"Lỗi khi tìm đường: {e}")
      path_coords = []

  # 4. Hiển thị bản đồ với đường đi được tìm thấy
  print("\n--- Đang hiển thị bản đồ ---")
  router.show_map(
      org=origin_coords,
      dests=[target_coords],
      route=path_coords,
  )

test_route_planner()

# Hệ thống tích hợp

### Khởi tạo mô hình

In [ ]:
route_planner = MapRouter()
traffic_estimator = TrafficEstimator()
disease_classifier = DiseaseClassifier()
patient_prioritizer = PatientPrioritizer()

print("=== Training traffic_estimator from dataset ===")
traffic_estimator.train(traffic_estimator_train_dataset)
print("\n=== Training disease_classifier from dataset ===")
disease_classifier.train(disease_classifier_train_dataset)
print("\n=== Loading knowledgebase for patient_prioritizer ===")
patient_prioritizer.load_knowledge_base("features/knowledgebase.txt")

print("\nDone ✅.")

### Cập nhật môi trường

Tuỳ chỉnh nếu muốn chạy với cấu hình môi trường khác (nằm trong cấu hình gợi ý).

In [ ]:
# Cấu hình môi trường
time_of_day = "Night"           # "Morning Peak" / "Afternoon" / "Evening Peak" / "Night"
day_of_week = "Weekday"         # "Weekday" / "Weekend"
weather_condition = "Rain"      # "Clear" / "Rain" / "Fog" / "Heatwave"

# Dự báo tình trạng giao thông
def weight_func(data, alpha = 0.5):
  road_type = data.get("highway")
  jam_level = traffic_estimator.estimate_traffic_density_level(time_of_day,
                                                          day_of_week,
                                                          weather_condition,
                                                          road_type)
  length = data.get("length")
  data["density_level"] = jam_level
  return length * (1 + alpha * jam_level)

route_planner.add_edges_attribute("weight", weight_func)


route_planner.show_map(show_density=True)

### Xử lý chính
Xử lý các yêu cầu xe cứu thương từ các bệnh nhân: hệ thống sẽ ưu tiên những bệnh nhân có triệu chứng nghiêm trọng nhất, sau đó là các bệnh nhân có bệnh được dự đoán nghiêm trọng nhất và cuối cùng là ưu tiên bệnh nhân gần nhất.

In [ ]:
def process_patients_requests(ambulance_coord: tuple[float, float], patients: list[Patient]):
  if not patients:
    print("Không có bệnh nhân để xử lý.")
    return []

  # Dự đoán bệnh của bệnh nhân
  for p in patients:
    p.predicted_disease = disease_classifier.predict(p.symptoms)

  # Danh sách các bệnh nhân được ưu tiên nhất dựa trên triệu chứng và bệnh
  most_prior_patients = patient_prioritizer.get_most_prioritized_patients(patients)

  # Tìm đường đi tới các bệnh nhân đó
  routes = [route_planner.optimal_path(ambulance_coord, p.position) for p in most_prior_patients]

  # Ưu tiên bệnh nhân gần nhất
  nearest = min(routes, key=lambda x: x[1])
  chosen_patient = most_prior_patients[routes.index(nearest)]
  path = nearest[0]

  prior_routes_info = {most_prior_patients[i]: routes[i][1] for i in range(len(most_prior_patients))}

  # Show kết quả
  print("\n=== Kết quả ===")
  print(f"\nVị trí xe cứu thương: {ambulance_coord}")
  print("Kết quả điều phối:")
  headers = ["ID", "Vị trí", "Triệu chứng", "Chẩn đoán", "Được ưu tiên", "Đường đi", "Được chọn"]
  rows = []
  for p in patients:
      is_prior = '*' if p in most_prior_patients else ''
      # Nếu nằm trong danh sách ưu tiên thì lấy chiều dài từ dictionary, format 2 chữ số thập phân
      route_len = f"{prior_routes_info[p]:.2f} m" if p in most_prior_patients else ""
      is_chosen = '*' if p is chosen_patient else ''

      rows.append([
          p.id,
          f"{p.position[0]:.4f}, {p.position[1]:.4f}", # Rút gọn hiển thị toạ độ cho bảng đẹp hơn
          ", ".join(p.symptoms),
          p.predicted_disease,
          is_prior,
          route_len,
          is_chosen
      ])
  print(tabulate(rows, headers=headers, tablefmt="grid"))
  print(f"\nĐường đi theo toạ độ: {path}")
  print("Đường đi trên bản đồ:")
  route_planner.show_map(route=path, org=ambulance_coord, dests=[p.position for p in patients])

  return path

### Giao diện tương tác

Thực hiện theo các bước sau:

1. Chọn vị trí xe cứu thương
2. Thêm một số lượng bệnh nhân tuỳ ý bằng cách lặp lại cách bước:
  - Chọn vị trí bệnh nhân
  - Liệt kê các triệu chứng
  - Nhấn "Thêm bệnh nhân"
3. Nhấn "Kết thúc và trả kết quả"

**Lưu ý**: Kết quả xuất ra màn hình có bao gồm hình ảnh bản đồ và việc render bản đồ có thể mất thời gian. Do đó, vui lòng kiên nhẫn chờ đợi hình ảnh cuối cùng.

In [ ]:
from modules.ui.dispatch_ui import *
create_dispatch_interface(route_planner.place, list(disease_classifier.symptoms_vocab), process_patients_requests)